In [1]:
import pandas as pd

# =========================
# 1. 파일 경로
# =========================
raw_file = "./data/공고원본_POSTING_ID포함_정리결과.xlsx"
classified_file = "./data/count_ge_10_classified.xlsx"
output_file = "normalized_freq_full_classified.xlsx"

# =========================
# 2. 데이터 불러오기
# =========================
raw_df = pd.read_excel(raw_file, sheet_name="normalized_freq")
high_df = pd.read_excel(classified_file, sheet_name="count_ge_10_classified")

# 컬럼 정리
raw_df["tech_stack"] = raw_df["tech_stack"].astype(str).str.strip()
high_df["tech_stack"] = high_df["tech_stack"].astype(str).str.strip()

raw_df["tech_stack_clean"] = raw_df["tech_stack"].str.lower().str.strip()
high_df["tech_stack_clean"] = high_df["tech_stack"].str.lower().str.strip()

# =========================
# 3. count >= 10 수작업 결과를 manual_map으로 만들기
# =========================
manual_map = dict(zip(high_df["tech_stack_clean"], high_df["category"]))

# =========================
# 4. 규칙 기반 자동 분류 함수
# =========================
def rule_based_category(stack: str) -> str:
    s = stack.lower().strip()

    # 1순위: 이미 수작업 분류된 키워드면 그대로 사용
    if s in manual_map:
        return manual_map[s]

    # -----------------
    # 백엔드
    # -----------------
    backend_keywords = [
        "java", "spring", "spring boot", "jsp", "servlet",
        "node", "node.js", "express",
        "django", "flask", "fastapi",
        "php", "laravel",
        "api", "rest", "graphql",
        "sql", "pl/sql",
        "oracle", "mysql", "mariadb", "postgres", "postgresql",
        "mssql", "db", "dbms", "rdbms",
        "jpa", "hibernate", "mybatis",
        "redis", "mongodb"
    ]

    # -----------------
    # 프론트
    # -----------------
    frontend_keywords = [
        "html", "css", "javascript", "typescript", "js", "ts",
        "react", "vue", "angular",
        "next", "next.js", "nuxt", "nuxt.js",
        "jquery",
        "react native", "flutter",
        "android", "ios", "kotlin", "swift",
        "figma", "ui", "ux"
    ]

    # -----------------
    # 인공지능
    # -----------------
    ai_keywords = [
        "python",
        "tensorflow", "pytorch", "keras",
        "scikit-learn", "sklearn",
        "pandas", "numpy", "matplotlib", "seaborn",
        "opencv",
        "nlp", "text mining", "textmining",
        "machine learning", "ml",
        "deep learning", "dl",
        "llm", "bert", "transformer",
        "langchain", "rag",
        "data analysis", "data mining",
        "통계", "회귀", "분류", "클러스터링"
    ]

    # -----------------
    # 인프라
    # -----------------
    infra_keywords = [
        "aws", "azure", "gcp", "cloud", "클라우드",
        "docker", "kubernetes", "k8s",
        "jenkins", "github actions", "ci/cd", "cicd",
        "linux", "unix",
        "terraform", "ansible",
        "nginx", "apache",
        "devops",
        "git", "github", "gitlab",
        "kafka", "hadoop", "spark",
        "etl", "airflow",
        "server", "network", "운영", "배포"
    ]

    # -----------------
    # 규칙 적용
    # 길이가 짧은 단어 오탐 줄이려고
    # 완전일치 먼저, 그 다음 포함 여부 확인
    # -----------------
    for kw in backend_keywords:
        if s == kw or kw in s:
            return "백엔드"

    for kw in frontend_keywords:
        if s == kw or kw in s:
            return "프론트"

    for kw in ai_keywords:
        if s == kw or kw in s:
            return "인공지능"

    for kw in infra_keywords:
        if s == kw or kw in s:
            return "인프라"

    # 끝까지 안 잡히면 임시 대표 배정
    # 여기서는 백엔드로 보내지 말고 검토 대상으로 남기기 위해 분류보류 사용
    return "분류보류"

# =========================
# 5. note 생성 함수
# =========================
def make_note(row) -> str:
    s = row["tech_stack_clean"]

    if s in manual_map:
        return "count >= 10 수작업 분류값 사용"

    if row["category"] == "분류보류":
        return "규칙 기반 자동 분류 실패, 수동 확인 필요"

    return "규칙 기반 자동 분류"

# =========================
# 6. 전체 분류 적용
# =========================
raw_df["category"] = raw_df["tech_stack_clean"].apply(rule_based_category)
raw_df["note"] = raw_df.apply(make_note, axis=1)

# =========================
# 7. 남은 항목(<10)만 따로 보기
# =========================
low_df = raw_df[raw_df["count"] < 10].copy()
low_unclassified_df = low_df[low_df["category"] == "분류보류"].copy()

# =========================
# 8. 저장
# =========================
result_df = raw_df.drop(columns=["tech_stack_clean"])

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    result_df.to_excel(writer, sheet_name="full_classified", index=False)
    low_df.to_excel(writer, sheet_name="count_lt_10_classified", index=False)
    low_unclassified_df.to_excel(writer, sheet_name="count_lt_10_unclassified", index=False)

print("저장 완료:", output_file)
print("\n카테고리별 개수")
print(result_df["category"].value_counts())

print("\ncount < 10 중 분류보류 개수")
print(len(low_unclassified_df))

저장 완료: normalized_freq_full_classified.xlsx

카테고리별 개수
category
분류보류    101
백엔드      35
프론트      35
인프라      23
인공지능     19
Name: count, dtype: int64

count < 10 중 분류보류 개수
101


In [2]:
import pandas as pd

raw_file = "./data/공고원본_POSTING_ID포함_정리결과.xlsx"
classified_file = "./data/count_ge_10_classified.xlsx"
output_file = "normalized_freq_full_classified_v2.xlsx"

# 1. 데이터 불러오기
raw_df = pd.read_excel(raw_file, sheet_name="normalized_freq")
high_df = pd.read_excel(classified_file, sheet_name="count_ge_10_classified")

raw_df["tech_stack"] = raw_df["tech_stack"].astype(str).str.strip()
high_df["tech_stack"] = high_df["tech_stack"].astype(str).str.strip()

raw_df["tech_stack_clean"] = raw_df["tech_stack"].str.lower().str.strip()
high_df["tech_stack_clean"] = high_df["tech_stack"].str.lower().str.strip()

# 2. count >= 10 수작업 결과
manual_map = dict(zip(high_df["tech_stack_clean"], high_df["category"]))

# 3. 현재 데이터셋용 exact_map
exact_map = {
    # 백엔드
    "c#": "백엔드",
    "c++": "백엔드",
    "c": "백엔드",
    "php": "백엔드",
    "mssql": "백엔드",
    "spring framework": "백엔드",
    "mybatis": "백엔드",
    "전자정부표준프레임워크": "백엔드",
    "fastapi": "백엔드",
    "ibatis": "백엔드",
    "nestjs": "백엔드",
    "db": "백엔드",
    "django": "백엔드",
    "jpa": "백엔드",
    "tibero": "백엔드",
    "flask": "백엔드",
    "json": "백엔드",
    "was": "백엔드",
    "xml": "백엔드",
    "servlet": "백엔드",
    ".net": "백엔드",
    "asp": "백엔드",
    "asp.net": "백엔드",
    "codeigniter": "백엔드",
    "go": "백엔드",
    "graphql": "백엔드",
    "groovy": "백엔드",
    "laravel": "백엔드",
    "powerbuilder": "백엔드",
    "swagger": "백엔드",
    "tomcat": "백엔드",
    ".net framework": "백엔드",
    "spring mvc": "백엔드",
    "visual c++": "백엔드",
    "vb": "백엔드",
    "vbscript": "백엔드",
    "tmax": "백엔드",
    "erp": "백엔드",
    "sfdc": "백엔드",
    "saas": "백엔드",
    "websocket": "백엔드",
    "nosql": "백엔드",
    "mongodb": "백엔드",
    "open source": "백엔드",

    # 프론트
    "ajax": "프론트",
    "vue.js": "프론트",
    "photoshop": "프론트",
    "react native": "프론트",
    "adobe xd": "프론트",
    "ui/ux": "프론트",
    "ios": "프론트",
    "webgl": "프론트",
    "nexacro": "프론트",
    "scss": "프론트",
    "three.js": "프론트",
    "unity": "프론트",
    "unreal": "프론트",
    "android studio": "프론트",
    "dart": "프론트",
    "illustrator": "프론트",
    "miplatform": "프론트",
    "recoil": "프론트",
    "redux": "프론트",
    "tailwindcss": "프론트",
    "aos": "프론트",
    "angular": "프론트",
    "chart.js": "프론트",
    "gsap": "프론트",
    "indesign": "프론트",
    "react query": "프론트",
    "sketch": "프론트",
    "swr": "프론트",
    "vite": "프론트",
    "websquare": "프론트",
    "xplatform": "프론트",
    "exbuilder6": "프론트",

    # 인공지능
    "ai": "인공지능",
    "머신러닝": "인공지능",
    "딥러닝": "인공지능",
    "llm": "인공지능",
    "opencv": "인공지능",
    "pandas": "인공지능",
    "r": "인공지능",
    "chatgpt": "인공지능",
    "rnn": "인공지능",
    "jax": "인공지능",
    "keras": "인공지능",
    "llm api": "인공지능",
    "lstm": "인공지능",
    "transformer": "인공지능",
    "gpt": "인공지능",
    "ai agent": "인공지능",
    "gemini": "인공지능",
    "hugging face": "인공지능",
    "numpy": "인공지능",
    "scikit-learn": "인공지능",
    "cnn": "인공지능",
    "cuda": "인공지능",
    "rag": "인공지능",
    "apache spark": "인공지능",
    "caffe": "인공지능",
    "chatgpt api": "인공지능",
    "claude": "인공지능",
    "etl": "인공지능",
    "gis tool": "인공지능",
    "langchain": "인공지능",
    "llmops": "인공지능",
    "llama": "인공지능",
    "matlab": "인공지능",
    "mcp": "인공지능",
    "nlp": "인공지능",
    "pruning": "인공지능",
    "sas": "인공지능",
    "scrapy": "인공지능",
    "slm": "인공지능",
    "stt": "인공지능",
    "tableau": "인공지능",
    "tableau prep": "인공지능",
    "tts": "인공지능",

    # 인프라
    "iot": "인프라",
    "jenkins": "인프라",
    "jira": "인프라",
    "microsoft office": "인프라",
    "network": "인프라",
    "kubernetes": "인프라",
    "apache": "인프라",
    "arduino": "인프라",
    "azure": "인프라",
    "eclipse": "인프라",
    "hadoop": "인프라",
    "unix": "인프라",
    "window": "인프라",
    "gcp": "인프라",
    "labview": "인프라",
    "rpa": "인프라",
    "raspberry pi": "인프라",
    "asana": "인프라",
    "assembly": "인프라",
    "ccie": "인프라",
    "ccna": "인프라",
    "ccnp": "인프라",
    "cisco": "인프라",
    "cafe24": "인프라",
    "cursor": "인프라",
    "discord": "인프라",
    "docker compose": "인프라",
    "elk": "인프라",
    "esp32": "인프라",
    "embedded linux": "인프라",
    "ga": "인프라",
    "gpio": "인프라",
    "hadoop-ecosystem": "인프라",
    "lidar": "인프라",
    "modern c++": "인프라",
    "ncp": "인프라",
    "nhn cloud": "인프라",
    "nginx": "인프라",
    "oci": "인프라",
    "rgb": "인프라",
    "ros2": "인프라",
    "selenium": "인프라",
    "shell": "인프라",
    "slack": "인프라",
    "spark": "인프라",
    "stm32": "인프라",
    "wireshark": "인프라"
}

# 4. fallback 규칙
def rule_based_category(stack: str) -> str:
    s = stack.lower().strip()

    if s in manual_map:
        return manual_map[s]

    if s in exact_map:
        return exact_map[s]

    backend_keywords = [
        "spring", "jsp", "servlet", "node", "express", "django", "flask",
        "fastapi", "php", "laravel", "api", "sql", "oracle", "mysql",
        "mariadb", "postgres", "mssql", "db", "jpa", "hibernate",
        "mybatis", "redis", "graphql", "tomcat", "swagger"
    ]

    frontend_keywords = [
        "html", "css", "javascript", "typescript", "react", "vue", "angular",
        "next", "nuxt", "jquery", "flutter", "android", "ios", "kotlin",
        "swift", "figma", "ui", "ux", "tailwind", "chart.js", "vite"
    ]

    ai_keywords = [
        "python", "tensorflow", "pytorch", "keras", "sklearn", "scikit",
        "pandas", "numpy", "opencv", "머신러닝", "딥러닝", "ai", "llm",
        "gpt", "rag", "langchain", "transformer", "cnn", "rnn", "nlp",
        "hugging face", "tableau", "sas", "stt", "tts"
    ]

    infra_keywords = [
        "aws", "azure", "gcp", "cloud", "클라우드", "docker", "kubernetes",
        "jenkins", "linux", "unix", "terraform", "ansible", "nginx",
        "apache", "devops", "git", "github", "kafka", "hadoop", "spark",
        "etl", "network", "shell", "wireshark", "oci", "ncp"
    ]

    for kw in backend_keywords:
        if s == kw or kw in s:
            return "백엔드"

    for kw in frontend_keywords:
        if s == kw or kw in s:
            return "프론트"

    for kw in ai_keywords:
        if s == kw or kw in s:
            return "인공지능"

    for kw in infra_keywords:
        if s == kw or kw in s:
            return "인프라"

    return "분류보류"

# 5. note
def make_note(row):
    s = row["tech_stack_clean"]

    if s in manual_map:
        return "count >= 10 수작업 분류"
    elif s in exact_map:
        return "현재 데이터셋 기준 exact_map 분류"
    elif row["category"] == "분류보류":
        return "추가 확인 필요"
    else:
        return "fallback 규칙 분류"

# 6. 적용
raw_df["category"] = raw_df["tech_stack_clean"].apply(rule_based_category)
raw_df["note"] = raw_df.apply(make_note, axis=1)

result_df = raw_df.drop(columns=["tech_stack_clean"])
unclassified_df = result_df[result_df["category"] == "분류보류"].copy()

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    result_df.to_excel(writer, sheet_name="full_classified", index=False)
    unclassified_df.to_excel(writer, sheet_name="unclassified", index=False)

print("저장 완료:", output_file)
print("\n카테고리별 개수")
print(result_df["category"].value_counts())
print("\n분류보류 개수:", len(unclassified_df))
print("\n분류보류 목록")
print(unclassified_df[["tech_stack", "count"]].sort_values("count", ascending=False))

저장 완료: normalized_freq_full_classified_v2.xlsx

카테고리별 개수
category
백엔드     59
인프라     55
인공지능    48
프론트     46
분류보류     5
Name: count, dtype: int64

분류보류 개수: 5

분류보류 목록
        tech_stack  count
83             CAD      4
93             풀스택      4
94              한컴      4
164  Google Sheets      1
207         VSCode      1
